Nombre: Ariel Huarachi Clemente   

Url git:https://github.com/Ariel-Huarachi/SIS420_IA_2_2025/tree/main/Laboratorio/Laboratorio6 

Importar las librerías

In [2]:

import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


In [3]:

class GTSRBDatasetCSV(Dataset):
    """
    Dataset que lee rutas y etiquetas desde un CSV.
    Se asume que el CSV tiene al menos dos columnas: una con la ruta relativa de la imagen
    (por defecto 'Path') y otra con el identificador de clase (por defecto 'ClassId').
    Ajusta los nombres de las columnas si tus CSV usan otros encabezados.
    """
    def __init__(self, csv_file, img_dir, transform=None, path_col='Path', label_col='ClassId'):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_rel_path = self.data.iloc[idx][self.path_col]
        img_path = os.path.join(self.img_dir, img_rel_path)
        image = Image.open(img_path)
        label = int(self.data.iloc[idx][self.label_col])
        if self.transform:
            image = self.transform(image)
        return image, label


Definir las transformaciones de los datos

Redimensionamos las imágenes a 32×32, aplicamos aumentos de datos en el conjunto de entrenamiento (flip horizontal, rotación, jitter de color) y normalizamos. Para el conjunto de validación solo redimensionamos y normalizamos

In [4]:

# Transformaciones para los conjuntos de entrenamiento y validación
ttrain_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

val_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Rutas a los CSV y directorios de imágenes
train_csv = 'data/Train.csv'
test_csv  = 'data/Test.csv'
train_img_dir = 'data/Train'
test_img_dir  = 'data/Test'

train_dataset = GTSRBDatasetCSV(train_csv, train_img_dir, transform=ttrain_transform, 
                               path_col='Path', label_col='ClassId')
val_dataset   = GTSRBDatasetCSV(test_csv, test_img_dir, transform=val_transform,
                               path_col='Path', label_col='ClassId')

# Crear DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size*2, shuffle=False, num_workers=2)

# Información básica sobre el dataset
num_classes = train_dataset.data['ClassId'].nunique()
print("Número de ejemplos de entrenamiento:", len(train_dataset))
print("Número de ejemplos de validación:", len(val_dataset))
print("Número de clases:", num_classes)


Número de ejemplos de entrenamiento: 39209
Número de ejemplos de validación: 12630
Número de clases: 43


In [5]:

# Definición de una red neuronal completamente conectada (MLP)

img_size = 32
num_channels = 3

# num_classes calculado en la celda anterior
def build_model(D_in=img_size*img_size*num_channels, H1=1024, H2=512, H3=256, D_out=num_classes):
    model = torch.nn.Sequential(
        torch.nn.Flatten(),
        torch.nn.Linear(D_in, H1),
        torch.nn.ReLU(),
        torch.nn.Linear(H1, H2),
        torch.nn.ReLU(),
        torch.nn.Linear(H2, H3),
        torch.nn.ReLU(),
        torch.nn.Linear(H3, D_out)
    )
    return model


In [6]:

# Función de entrenamiento con early stopping
def fit(model, train_loader, val_loader, optimizer, epochs=50, log_each=5, early_stopping=10, device=None):
    criterion = torch.nn.CrossEntropyLoss()
    history = {'epoch': [], 'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}

    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    best_val_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        # Entrenamiento
        model.train()
        train_losses, train_accuracies = [], []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            preds = torch.argmax(outputs, dim=1)
            train_accuracies.append(accuracy_score(y_batch.cpu().numpy(), preds.cpu().numpy()))
        avg_train_loss = np.mean(train_losses)
        avg_train_acc = np.mean(train_accuracies)

        # Validación
        model.eval()
        val_losses, val_accuracies = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_losses.append(loss.item())
                preds = torch.argmax(outputs, dim=1)
                val_accuracies.append(accuracy_score(y_batch.cpu().numpy(), preds.cpu().numpy()))
        avg_val_loss = np.mean(val_losses)
        avg_val_acc = np.mean(val_accuracies)

        # Registro de métricas
        history['epoch'].append(epoch)
        history['loss'].append(avg_train_loss)
        history['acc'].append(avg_train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(avg_val_acc)

        if epoch % log_each == 0:
            print(f"Epoch {epoch}/{epochs} | loss {avg_train_loss:.4f} acc {avg_train_acc:.4f} | val_loss {avg_val_loss:.4f} val_acc {avg_val_acc:.4f}")

        # Mejor modelo
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), 'best_model_csv.pt')
            print(f"Nueva mejor precisión de validación: {best_val_acc:.4f} en la época {epoch}")
        else:
            epochs_without_improvement += 1

        # Early stopping
        if early_stopping and epochs_without_improvement > early_stopping:
            print(f"Entrenamiento detenido en la época {epoch} — mejor val_acc: {best_val_acc:.4f}")
            break

    # Cargar el mejor modelo
    model.load_state_dict(torch.load('best_model_csv.pt'))
    return history


In [ ]:

optimizers = {
    'SGD': lambda params: torch.optim.SGD(params, lr=0.01),
    'Momentum': lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9),
    'RMSprop': lambda params: torch.optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: torch.optim.Adam(params, lr=0.001)
}

histories = {}

# Entrenamiento para cada optimizador
for name, opt_func in optimizers.items():
    print(f"\nEntrenando con el optimizador {name}...\n")
    model = build_model()                     # Aquí la función ya debe estar definida
    optimizer = opt_func(model.parameters())
    histories[name] = fit(model, train_loader, val_loader, optimizer,
                         
    print(f"Entrenamiento con {name} completado.\n")
    




Entrenando con el optimizador SGD...



In [ ]:

# Graficar precisión y pérdida
plt.figure(figsize=(12, 5))

# Precisión
plt.subplot(1, 2, 1)
for name, history in histories.items():
    plt.plot(history['epoch'], history['acc'], label=f'{name} (train)')
    plt.plot(history['epoch'], history['val_acc'], linestyle='--', label=f'{name} (val)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.title('Precisión de Entrenamiento vs Validación')
plt.legend()

# Pérdida
plt.subplot(1, 2, 2)
for name, history in histories.items():
    plt.plot(history['epoch'], history['loss'], label=f'{name} (train)')
    plt.plot(history['epoch'], history['val_loss'], linestyle='--', label=f'{name} (val)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Pérdida de Entrenamiento vs Validación')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

# Evaluación final del mejor modelo cargado
best_model = build_model()
best_model.load_state_dict(torch.load('best_model_csv.pt'))
best_model.eval()

correct = 0
count = 0
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = best_model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        count += y_batch.size(0)

print(f"Precisión final en el conjunto de validación: {correct / count:.4f}")
